In [1]:

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [2]:
# 1. Try Modern Small Models
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
model = model.to(device)

print(f"{model_name} loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama/TinyLlama-1.1B-Chat-v1.0 loaded successfully!


In [3]:
# 2. Chat Format (Very Important)
def generate_response(prompt, max_new_tokens=300, temperature=0.7):
    messages = [
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test
print(generate_response("Tell me a short funny story about an Ethiopian programmer."))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|>
Tell me a short funny story about an Ethiopian programmer.
<|assistant|>
There was a young Ethiopian programmer named Amara who had always been fascinated by coding. As a child, she would spend hours practicing and experimenting with various programming languages, always dreaming of becoming a professional programmer someday.

Amara's love for coding took a backseat to her studies in computer science when she moved to the United States for college. However, her passion for coding never faded, and she continued to hone her skills and learn new languages.

One day, while working on a project for her job, Amara stumbled upon a bug in the code. She spent hours debugging and fixing the issue, but it was a long and tedious process. Finally, she was able to fix the bug, but the project manager was not impressed.

"You're not a programmer," the project manager said, scowling. "You're a programmer for a living."

Amara was shocked. She had never heard such a thing before. "But I've bee

In [6]:
# 3. Mini Project: AI Assistant
print("=== AI Assistant ===")

def ai_assistant(query):
    prompt = f"""You are a helpful AI assistant. Answer the following question clearly and concisely.

Question: {query}"""

    return generate_response(prompt, max_new_tokens=250, temperature=0.6)

# Test the assistant
questions = [
    "What are the best career options in AI in 2026?",
    "Give me 5 Ethiopian food recipes that are easy to make.",
    "Explain transformers in simple terms."
]

for q in questions:
    print(f"\nQ: {q}")
    print("A:", ai_assistant(q)[:400] + "...")

[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== AI Assistant ===

Q: What are the best career options in AI in 2026?


[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: <|user|>
You are a helpful AI assistant. Answer the following question clearly and concisely.

Question: What are the best career options in AI in 2026?
<|assistant|>
As per a report by Gartner, the demand for artificial intelligence (AI) professionals is expected to grow significantly in 2026, with the following top career options:

1. Data Scientist
2. Machine Learning Engineer
3. AI Architect
4...

Q: Give me 5 Ethiopian food recipes that are easy to make.


[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: <|user|>
You are a helpful AI assistant. Answer the following question clearly and concisely.

Question: Give me 5 Ethiopian food recipes that are easy to make.
<|assistant|>
Sure! Here are five easy and delicious Ethiopian food recipes to try:

1. Bishoftu Doro Wat (Ethiopian Lentil Soup with Chickpeas)
Ingredients:
- 2 cups dried chickpeas
- 1 onion, chopped
- 2 garlic cloves, minced
- 2 carrots...

Q: Explain transformers in simple terms.
A: <|user|>
You are a helpful AI assistant. Answer the following question clearly and concisely.

Question: Explain transformers in simple terms.
<|assistant|>
Transformers are a family of neural network models developed by researchers at the University of Toronto. They are known for their ability to learn and generalize complex tasks from limited training data, making them a popular choice for tasks...
